# 01 · Results tables (camera-ready)

Running this notebook top to bottom **writes every table of the paper** to
`artifacts/tables/tex/*.tex` (LaTeX, with the paper's `\label`) and
`artifacts/tables/csv/*.csv` (mean, std, n_seeds per cell), plus the audits in
`artifacts/checks/`. Each table is one function in
`wavelet_flow_matching/analysis/paper_tables.py`; `scripts/reproduce_tables.py` calls the
same functions, so notebook and script produce byte-identical files.

Every cell = mean ± std over inference seeds 42 / 84 / 168 (deterministic
baselines: one run). Spectral coherence "averaged over the rollout" is
horizon-balanced (Table 1, S14); the physical-parameter tables (Table 3,
S15–S19) average over all lead times. Horizon bins are inclusive.

**Mean ± std or mean only, per table.** Every table cell below passes
`show_std=`:
* `True`  → cells read "mean ± std" over the three seeds;
* `False` → cells show the seed mean only; the caption states that the
  across-seed std is below 5% of the mean (`T.STD_REL`) and marks with * every
  cell where it is not. Sentence and marks are generated from the data.

Table 3a/3b (`show_std_main`) and S11 (`show_std_s11`) have their own flags.
**Ranking:** bold = best, underline = second best per column, ranked on the
unrounded means, so exactly one of each (`T.TIE_MODE = "shared"` to let printed
ties share the mark). Files are overwritten in place, so the paper's `\input`
paths do not change; the CSVs always keep the std.

File names start with the table number in the camera-ready paper (`T` = main text, `S` = appendix).

| Section | Paper | Function | Files |
|---|---|---|---|
| 1 | Table 1, S11 | `t1_wavelet_selection` | `T1_wavelet_selection`, `S11_optimal_wavelets` |
| 2 | Table 2, S23, S24 | `t2_s23_s24_profiling` | `T2_inference_speedup`, `S23_profiling_selected`, `S24_profiling_all_variants` |
| 3 | Table 3a, S15 | `s15_crps_tcool` | `T3a_crps_tcool`, `S15_crps_tcool` |
| 3 | Table 3b, S17, S18 | `s17_crps_ra`, `s18_crps_pr` | `T3b_crps_Ra`, `S17_crps_Ra`, `S18_crps_Pr` |
| 4 | S12, S13, S14 | `s12_vrmse`, `s13_crps`, `s14_spectral_bands` | same names |
| 5 | S16, S19 | `s16_spectral_tcool`, `s19a_spectral_ra`, `s19b_spectral_pr` | same names |
| 6 | S20, S21, S22, S2 | `s20_ke_mae`, `s21_s22_ic_perturbation`, `s2_cross_scale` | `S20_ke_mae`, `S21_ic_crps`, `S22_ic_spectral`, `S2_cross_scale_sensitivity` |
| 6 | not in paper | `x_rb_latentfm` | `X_rb_latentfm` |
| 7 | checks | `write_seed_spread`, `checks.paper_reproduction` | `artifacts/checks/` |


In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display, Markdown
from wavelet_flow_matching.analysis import catalog as C, tables as T, paper_tables as P
from wavelet_flow_matching.analysis.checks import paper_reproduction, summary
pd.set_option("display.max_columns", 40, "display.width", 250)

def view(res, rename_cols=None):
    """Display a (mean, std, n) result as 'mean ± std' with readable labels."""
    mean, std = res[0], res[1]
    out = T.show(mean, std)
    if rename_cols:
        out = out.rename(columns=rename_cols)
    return out

# ── Table options (apply to every table) ─────────────────────────────────────
# Per table, `show_std=` in its cell chooses "mean ± std" (True) or mean only (False).
# Mean-only tables: caption says the across-seed std is below STD_REL of the mean,
# and cells where it is not get a * (both generated from the data).
T.STD_REL  = 0.05          # 5 % of the mean
# Bold = best, underline = second best, per column:
#   "unrounded": ranked on full-precision means -> exactly one of each per column
#   "shared"   : ranked on printed (3-decimal) values -> printed ties share the mark
T.TIE_MODE = "unrounded"

df, gdf = P.load_all()     # per-field metric rows, `_global_` physics rows

loaded 856,572 rows from 111 files (736,308 per-field, 120,264 global)


## 1 · Table 1 / S11 — wavelet selection

In [2]:
t1 = P.t1_wavelet_selection(df, show_std=False, show_std_s11=False)   # Table 1 & Table S11
display(pd.read_csv(T.TABLES_CSV / "S11_optimal_wavelets.csv"))
print((T.TABLES_TEX / "S11_optimal_wavelets.tex").read_text())
view(t1, C.PDE_LABELS)

  wrote T1_wavelet_selection
  wrote S11_optimal_wavelets


,pde,scale,wavelet,mean,std
0,turbulent_radiative_layer_2D,1scale,haar,0.147050,0.000208
1,turbulent_radiative_layer_2D,3scale,haar,0.149129,0.000277
2,rayleigh_benard,1scale,db4,0.121084,0.001956
3,rayleigh_benard,3scale,db4,0.109106,0.001077
4,active_matter,1scale,haar,0.241397,0.000665
5,active_matter,3scale,db2,0.219940,0.000322


\begin{table}[ht]
\centering
\caption{Optimal mother wavelet per WFM variant and PDE system, selected by lowest spectral coherence RMSE averaged across \textit{Low}, \textit{Mid}, and \textit{High} frequency bands and rollout horizons (Table~\ref{tab:avg_sc_rmse_gen_wavelets}). In parentheses: the selected wavelet's score; the selection is made on the seed mean. Mean over three inference seeds (42, 84, 168); the standard deviation across seeds is below 5\% of the mean for all entries.}
\label{si-tab:optimal_wavelets}
\begin{tabular}{lccc}
\toprule
 & Turbulent Radiative Layer 2D & Rayleigh--B\'{e}nard & Active Matter \\
\midrule
WFM 1-scale & $\mathrm{haar}$ (0.147) & $\mathrm{db}4$ (0.121) & $\mathrm{haar}$ (0.241) \\
WFM 3-scale & $\mathrm{haar}$ (0.149) & $\mathrm{db}4$ (0.109) & $\mathrm{db}2$ (0.220) \\
\bottomrule
\end{tabular}
\end{table}



,Turbulent Radiative Layer 2D,Rayleigh--B\'{e}nard,Active Matter
WFM-1 haar,0.147 ± 0.0002,0.140 ± 0.0008,0.241 ± 0.0007
WFM-1 db2,0.154 ± 0.0002,0.132 ± 0.0007,0.288 ± 0.0001
WFM-1 db4,0.149 ± 0.0002,0.121 ± 0.0020,0.303 ± 0.0002
WFM-1 db6,0.150 ± 0.0003,0.140 ± 0.0025,0.290 ± 0.0001
WFM-3 haar,0.149 ± 0.0003,0.131 ± 0.0007,0.247 ± 0.0007
WFM-3 db2,0.154 ± 0.0011,0.111 ± 0.0013,0.220 ± 0.0003
WFM-3 db4,0.154 ± 0.0004,0.109 ± 0.0011,0.249 ± 0.0002
WFM-3 db6,0.153 ± 0.0005,0.112 ± 0.0005,0.238 ± 0.0003
FM$_\text{pixel}$,0.153 ± 0.0008,0.131 ± 0.0007,0.244 ± 0.0005
FM$_\text{fourier}$,0.379 ± 0.0004,0.266 ± 0.0004,0.376 ± 0.0005


## 2 · Table 2 / S23 / S24 — inference profiling


In [3]:
prof = P.t2_s23_s24_profiling()
print((T.TABLES_TEX / "T2_inference_speedup.tex").read_text())
from wavelet_flow_matching.analysis.profiling import selected
selected(prof).round(3)

  wrote T2/S18/S19 profiling
\begin{tabular}{lcc}
\toprule
\textbf{PDE} & \textbf{WFM 1-scale} & \textbf{WFM 3-scale} \\
\midrule
TRL & 1.25$\times$ & 1.14$\times$ \\
RB & 1.52$\times$ & 1.38$\times$ \\
AM & 1.46$\times$ & 1.26$\times$ \\
\bottomrule
\end{tabular}



,pde,model,wall_clock_s,sec_per_step,gpu_mem_GiB,throughput_Mfr_s,speedup,mem_ratio
1,turbulent_radiative_layer_2D,FM_pixel,247.75,0.035,0.590,1.386,1.000,1.000
5,turbulent_radiative_layer_2D,WFM_1scale_haar,198.06,0.028,0.591,1.733,1.251,0.998
9,turbulent_radiative_layer_2D,WFM_3scale_haar,218.27,0.031,0.663,1.573,1.135,0.890
11,rayleigh_benard,FM_pixel,633.85,0.045,1.441,1.459,1.000,1.000
13,rayleigh_benard,WFM_1scale_db4,418.13,0.030,1.444,2.212,1.516,0.998
17,rayleigh_benard,WFM_3scale_db4,457.89,0.032,1.515,2.020,1.384,0.951
21,active_matter,FM_pixel,310.77,0.046,1.629,1.429,1.000,1.000
25,active_matter,WFM_1scale_haar,213.28,0.031,1.636,2.082,1.457,0.996
26,active_matter,WFM_3scale_db2,247.36,0.037,1.713,1.795,1.256,0.951


## 3 · Table 3 / S15 / S17 / S18 — CRPS per physical parameter

In [4]:
s13 = P.s15_crps_tcool(df, show_std=False, show_std_main=False)  # Table S15 & Table 3a
view(s13, lambda v: f"tcool={v:g}")

  wrote S15_crps_tcool


  wrote T3a_crps_tcool


,tcool=0.03,tcool=0.06,tcool=0.1,tcool=0.18,tcool=0.32,tcool=0.56,tcool=1,tcool=1.78,tcool=3.16
WFM-1 haar,1.165 ± 0.0431,1.135 ± 0.0566,0.865 ± 0.0525,1.044 ± 0.0282,0.830 ± 0.0515,0.798 ± 0.0303,0.806 ± 0.0564,0.983 ± 0.0362,0.673 ± 0.0232
WFM-1 db2,1.112 ± 0.0306,1.078 ± 0.0248,0.878 ± 0.0419,1.057 ± 0.0397,0.880 ± 0.0654,0.779 ± 0.0152,0.853 ± 0.0528,0.976 ± 0.0506,1.002 ± 0.0606
WFM-1 db4,1.170 ± 0.0484,1.102 ± 0.0438,0.847 ± 0.0435,1.084 ± 0.0521,0.875 ± 0.0047,0.849 ± 0.0193,0.879 ± 0.0648,0.946 ± 0.0382,0.719 ± 0.0356
WFM-1 db6,1.076 ± 0.0233,1.074 ± 0.0145,0.831 ± 0.0205,1.041 ± 0.0172,0.823 ± 0.0212,0.773 ± 0.0340,0.741 ± 0.0162,0.943 ± 0.0180,0.594 ± 0.0398
WFM-3 haar,1.107 ± 0.0118,1.117 ± 0.0341,0.868 ± 0.0187,0.988 ± 0.0665,0.773 ± 0.0204,0.765 ± 0.0458,0.675 ± 0.0260,0.961 ± 0.0109,0.595 ± 0.0245
WFM-3 db2,1.248 ± 0.0241,1.141 ± 0.0438,0.894 ± 0.0425,1.027 ± 0.0250,0.836 ± 0.0171,0.783 ± 0.0407,0.751 ± 0.0401,1.019 ± 0.0085,0.656 ± 0.0276
WFM-3 db4,1.048 ± 0.0192,1.006 ± 0.0250,0.817 ± 0.0417,0.982 ± 0.0486,0.817 ± 0.0330,0.749 ± 0.0267,0.692 ± 0.0278,0.906 ± 0.0049,0.632 ± 0.0111
WFM-3 db6,1.075 ± 0.0177,1.034 ± 0.0150,0.882 ± 0.0383,1.000 ± 0.0529,0.863 ± 0.0385,0.850 ± 0.0268,0.770 ± 0.0476,0.969 ± 0.0384,0.617 ± 0.0063
FM$_\text{pixel}$,1.220 ± 0.0400,1.126 ± 0.0159,0.875 ± 0.0447,1.061 ± 0.0475,0.855 ± 0.0461,0.783 ± 0.0039,0.731 ± 0.0121,0.978 ± 0.0213,0.551 ± 0.0303
FM$_\text{fourier}$,1.439 ± 0.0085,1.424 ± 0.0131,1.195 ± 0.0060,1.419 ± 0.0049,1.278 ± 0.0064,1.226 ± 0.0235,1.170 ± 0.0142,1.347 ± 0.0131,1.032 ± 0.0163


In [5]:
s15 = P.s17_crps_ra(df, show_std=False, show_std_main=False)  # Table S17 & Table 3b
view(s15, lambda v: C.PARAM_LABELS["Rayleigh"](v))

  wrote S17_crps_Ra
  wrote T3b_crps_Ra


,$10^{6}$,$10^{8}$,$10^{10}$
WFM-1 haar,0.059 ± 0.0013,0.060 ± 0.0051,0.051 ± 0.0012
WFM-1 db2,0.052 ± 0.0018,0.062 ± 0.0032,0.052 ± 0.0029
WFM-1 db4,0.059 ± 0.0076,0.060 ± 0.0027,0.051 ± 0.0013
WFM-1 db6,0.060 ± 0.0043,0.069 ± 0.0041,0.075 ± 0.0011
WFM-3 haar,0.060 ± 0.0045,0.060 ± 0.0027,0.055 ± 0.0028
WFM-3 db2,0.054 ± 0.0044,0.055 ± 0.0014,0.049 ± 0.0007
WFM-3 db4,0.053 ± 0.0050,0.058 ± 0.0021,0.048 ± 0.0028
WFM-3 db6,0.065 ± 0.0064,0.055 ± 0.0021,0.057 ± 0.0007
FM$_\text{pixel}$,0.057 ± 0.0075,0.066 ± 0.0042,0.051 ± 0.0023
FM$_\text{fourier}$,0.071 ± 0.0012,0.081 ± 0.0001,0.065 ± 0.0006


In [6]:
s16 = P.s18_crps_pr(df, show_std=False)        # Table S18
view(s16, lambda v: C.PARAM_LABELS["Prandtl"](v))

  wrote S18_crps_Pr


,$10^{-1}$,$1$,$10^{1}$
WFM-1 haar,0.091 ± 0.0009,0.050 ± 0.0042,0.029 ± 0.0009
WFM-1 db2,0.088 ± 0.0052,0.049 ± 0.0021,0.030 ± 0.0005
WFM-1 db4,0.088 ± 0.0066,0.048 ± 0.0017,0.034 ± 0.0007
WFM-1 db6,0.087 ± 0.0028,0.066 ± 0.0039,0.051 ± 0.0009
WFM-3 haar,0.092 ± 0.0067,0.050 ± 0.0020,0.033 ± 0.0001
WFM-3 db2,0.079 ± 0.0032,0.048 ± 0.0006,0.031 ± 0.0006
WFM-3 db4,0.079 ± 0.0051,0.051 ± 0.0040,0.028 ± 0.0007
WFM-3 db6,0.086 ± 0.0066,0.050 ± 0.0018,0.040 ± 0.0014
FM$_\text{pixel}$,0.088 ± 0.0080,0.054 ± 0.0013,0.033 ± 0.0010
FM$_\text{fourier}$,0.114 ± 0.0009,0.058 ± 0.0006,0.045 ± 0.0002


## 4 · S12 / S13 / S14 — VRMSE, CRPS per horizon; spectral coherence per band

In [7]:
view(P.s12_vrmse(df, show_std=False))                 # Table S12

  wrote S12_vrmse


turbulent_radiative_layer_2D                                  rayleigh_benard                                    active_matter                                
                                            01:32           33:64           65:97            01:32           33:64          65:196            01:16           17:32           33:77
WFM-1 haar                         0.544 ± 0.0110  0.791 ± 0.0163  0.789 ± 0.0137   1.782 ± 0.0404  2.020 ± 0.0489  0.788 ± 0.0231   2.847 ± 0.0151  1.541 ± 0.0108  1.079 ± 0.0036
WFM-1 db2                          0.550 ± 0.0031  0.750 ± 0.0105  0.850 ± 0.0237   1.333 ± 0.0457  1.844 ± 0.1027  0.791 ± 0.0096   5.652 ± 0.0194  2.930 ± 0.0041  1.622 ± 0.0031
WFM-1 db4                          0.550 ± 0.0071  0.730 ± 0.0072  0.811 ± 0.0011   1.689 ± 0.0160  2.271 ± 0.0771  0.802 ± 0.0146  14.227 ± 0.0271  5.574 ± 0.0106  2.488 ± 0.0014
WFM-1 db6                          0.536 ± 0.0048  0.703 ± 0.0088  0.786 ± 0.0173   1.441 ± 0.0124  1.949 ± 0.0278  1.025 ± 0.0193   6.732 ± 0.0089  2.927 ± 0.0118  1.553 ± 0.0096
WFM-3 haar                         0.549 ± 0.0075  0.824 ± 0.0172  0.797 ± 0.0145   1.559 ± 0.0320  2.318 ± 0.0948  0.807 ± 0.0166   0.848 ± 0.0083  1.327 ± 0.0149  1.022 ± 0.0017
WFM-3 db2                          0.655 ± 0.0058  0.893 ± 0.0162  0.981 ± 0.0218   1.051 ± 0.0432  1.729 ± 0.0479  0.766 ± 0.0075   0.933 ± 0.0109  1.349 ± 0.0327  1.085 ± 0.0067
WFM-3 db4                          0.586 ± 0.0155  0.759 ± 0.0069  0.803 ± 0.0057   1.029 ± 0.0586  1.869 ± 0.0478  0.770 ± 0.0091   3.242 ± 0.0133  2.271 ± 0.0205  1.418 ± 0.0045
WFM-3 db6                          0.591 ± 0.0049  0.879 ± 0.0068  0.831 ± 0.0190   0.755 ± 0.0255  1.598 ± 0.0595  0.834 ± 0.0177   2.341 ± 0.0096  2.118 ± 0.0121  1.276 ± 0.0063
FM$_\text{pixel}$                  0.574 ± 0.0020  0.891 ± 0.0098  0.830 ± 0.0161   1.319 ± 0.0300  1.962 ± 0.0303  0.821 ± 0.0245   0.580 ± 0.0038  1.380 ± 0.0183  1.028 ± 0.0016
FM$_\text{fourier}$                0.619 ± 0.0156  0.771 ± 0.0163  0.782 ± 0.0258   1.098 ± 0.0158  0.810 ± 0.0052  0.839 ± 0.0017   1.302 ± 0.0007  1.135 ± 0.0052  1.027 ± 0.0041
WDNO                               1.517 ± 0.0001  1.834 ± 0.0010  1.953 ± 0.0026  19.262 ± 0.0116  8.171 ± 0.0320  1.392 ± 0.0020   4.475 ± 0.0109  2.181 ± 0.0019  1.400 ± 0.0000
LatentFM$_{\div 64}$                           --              --              --   9.515 ± 0.0422  4.080 ± 0.1018  0.917 ± 0.0202               --              --              --
WNO                                           inf             inf              --           44.226          24.463           2.452           10.669           4.321           2.002
FNO                                         2.952           4.978           4.349            3.368           3.972             inf            1.682           3.227           1.727
TFNO                                        2.300           6.534           6.387          102.678          31.842           2.965            4.443           2.968           1.525

In [8]:
view(P.s13_crps(df, show_std=False))                  # Table S13

  wrote S13_crps


turbulent_radiative_layer_2D                                 rayleigh_benard                                   active_matter                                
                                            01:32           33:64           65:97           01:32           33:64          65:196           01:16           17:32           33:77
WFM-1 haar                         0.492 ± 0.0062  1.047 ± 0.0334  1.218 ± 0.0323  0.005 ± 0.0000  0.037 ± 0.0004  0.074 ± 0.0029  0.119 ± 0.0007  0.190 ± 0.0010  0.198 ± 0.0007
WFM-1 db2                          0.447 ± 0.0069  1.097 ± 0.0077  1.316 ± 0.0324  0.005 ± 0.0001  0.036 ± 0.0005  0.073 ± 0.0024  0.173 ± 0.0003  0.239 ± 0.0002  0.261 ± 0.0005
WFM-1 db4                          0.521 ± 0.0062  1.052 ± 0.0024  1.241 ± 0.0209  0.007 ± 0.0000  0.041 ± 0.0003  0.073 ± 0.0032  0.326 ± 0.0000  0.388 ± 0.0002  0.387 ± 0.0000
WFM-1 db6                          0.451 ± 0.0026  0.983 ± 0.0169  1.188 ± 0.0079  0.005 ± 0.0000  0.040 ± 0.0001  0.090 ± 0.0013  0.175 ± 0.0001  0.222 ± 0.0001  0.231 ± 0.0005
WFM-3 haar                         0.410 ± 0.0033  1.018 ± 0.0108  1.178 ± 0.0297  0.004 ± 0.0001  0.037 ± 0.0004  0.077 ± 0.0038  0.080 ± 0.0002  0.169 ± 0.0033  0.191 ± 0.0006
WFM-3 db2                          0.465 ± 0.0078  1.133 ± 0.0061  1.179 ± 0.0256  0.003 ± 0.0000  0.032 ± 0.0003  0.069 ± 0.0016  0.087 ± 0.0009  0.176 ± 0.0030  0.196 ± 0.0021
WFM-3 db4                          0.427 ± 0.0041  0.984 ± 0.0072  1.130 ± 0.0164  0.004 ± 0.0001  0.032 ± 0.0003  0.069 ± 0.0011  0.109 ± 0.0014  0.209 ± 0.0009  0.239 ± 0.0001
WFM-3 db6                          0.451 ± 0.0058  1.026 ± 0.0065  1.200 ± 0.0150  0.004 ± 0.0000  0.031 ± 0.0006  0.079 ± 0.0035  0.100 ± 0.0013  0.197 ± 0.0005  0.213 ± 0.0010
FM$_\text{pixel}$                  0.418 ± 0.0064  1.088 ± 0.0066  1.210 ± 0.0172  0.005 ± 0.0000  0.036 ± 0.0008  0.076 ± 0.0047  0.059 ± 0.0005  0.166 ± 0.0005  0.194 ± 0.0005
FM$_\text{fourier}$                0.857 ± 0.0014  1.418 ± 0.0020  1.560 ± 0.0109  0.009 ± 0.0001  0.038 ± 0.0001  0.096 ± 0.0004  0.185 ± 0.0005  0.227 ± 0.0022  0.218 ± 0.0010
WDNO                               2.460 ± 0.0018  3.374 ± 0.0036  4.284 ± 0.0130  0.050 ± 0.0000  0.102 ± 0.0002  0.138 ± 0.0003  0.218 ± 0.0001  0.245 ± 0.0000  0.266 ± 0.0000
LatentFM$_{\div 64}$                           --              --              --  0.026 ± 0.0001  0.057 ± 0.0003  0.088 ± 0.0042              --              --              --

In [9]:
view(P.s14_spectral_bands(df, show_std=False))        # Table S14

  wrote S14_spectral_bands


turbulent_radiative_layer_2D                                 rayleigh_benard                                   active_matter                                
                                              Low             Mid            High             Low             Mid            High             Low             Mid            High
WFM-1 haar                         0.132 ± 0.0002  0.175 ± 0.0006  0.134 ± 0.0002  0.171 ± 0.0023  0.132 ± 0.0002  0.116 ± 0.0012  0.229 ± 0.0017  0.291 ± 0.0003  0.204 ± 0.0000
WFM-1 db2                          0.139 ± 0.0005  0.181 ± 0.0004  0.142 ± 0.0003  0.167 ± 0.0015  0.126 ± 0.0008  0.105 ± 0.0014  0.291 ± 0.0002  0.336 ± 0.0003  0.236 ± 0.0001
WFM-1 db4                          0.132 ± 0.0002  0.175 ± 0.0003  0.140 ± 0.0002  0.168 ± 0.0049  0.118 ± 0.0009  0.077 ± 0.0002  0.306 ± 0.0006  0.351 ± 0.0001  0.251 ± 0.0001
WFM-1 db6                          0.133 ± 0.0006  0.175 ± 0.0003  0.141 ± 0.0002  0.196 ± 0.0034  0.131 ± 0.0015  0.093 ± 0.0025  0.283 ± 0.0005  0.346 ± 0.0002  0.242 ± 0.0001
WFM-3 haar                         0.133 ± 0.0009  0.174 ± 0.0002  0.141 ± 0.0003  0.149 ± 0.0011  0.130 ± 0.0008  0.114 ± 0.0008  0.205 ± 0.0010  0.280 ± 0.0012  0.256 ± 0.0004
WFM-3 db2                          0.139 ± 0.0018  0.188 ± 0.0013  0.136 ± 0.0005  0.141 ± 0.0029  0.113 ± 0.0008  0.079 ± 0.0008  0.199 ± 0.0011  0.262 ± 0.0001  0.198 ± 0.0001
WFM-3 db4                          0.141 ± 0.0012  0.178 ± 0.0001  0.144 ± 0.0001  0.143 ± 0.0020  0.109 ± 0.0009  0.075 ± 0.0006  0.219 ± 0.0010  0.292 ± 0.0004  0.235 ± 0.0001
WFM-3 db6                          0.136 ± 0.0005  0.175 ± 0.0010  0.148 ± 0.0002  0.147 ± 0.0007  0.110 ± 0.0003  0.080 ± 0.0006  0.196 ± 0.0007  0.258 ± 0.0001  0.260 ± 0.0002
FM$_\text{pixel}$                  0.141 ± 0.0015  0.181 ± 0.0012  0.138 ± 0.0009  0.149 ± 0.0007  0.129 ± 0.0004  0.115 ± 0.0013  0.183 ± 0.0017  0.258 ± 0.0004  0.291 ± 0.0002
FM$_\text{fourier}$                0.235 ± 0.0005  0.487 ± 0.0004  0.415 ± 0.0006  0.327 ± 0.0011  0.263 ± 0.0001  0.209 ± 0.0002  0.426 ± 0.0014  0.463 ± 0.0002  0.238 ± 0.0000
WDNO                               0.199 ± 0.0004  0.249 ± 0.0002  0.366 ± 0.0003  0.392 ± 0.0007  0.474 ± 0.0002  0.535 ± 0.0001  0.367 ± 0.0007  0.345 ± 0.0001  0.200 ± 0.0000
LatentFM$_{\div 64}$                           --              --              --  0.196 ± 0.0002  0.159 ± 0.0003  0.131 ± 0.0005              --              --              --

## 5 · S16 / S19 — spectral coherence per band × physical parameter

In [10]:
view(P.s16_spectral_tcool(df, show_std=False))        # Table S16

  wrote S16_spectral_tcool


Low                                                                                                                                             Mid                                                                  \
                               0.03            0.06            0.10            0.18            0.32            0.56            1.00            1.78            3.16            0.03            0.06            0.10            0.18            0.32   
WFM-1 haar           0.138 ± 0.0066  0.139 ± 0.0032  0.138 ± 0.0026  0.138 ± 0.0035  0.132 ± 0.0006  0.125 ± 0.0036  0.127 ± 0.0046  0.136 ± 0.0026  0.121 ± 0.0052  0.190 ± 0.0014  0.194 ± 0.0011  0.189 ± 0.0005  0.189 ± 0.0010  0.180 ± 0.0003   
WFM-1 db2            0.141 ± 0.0020  0.139 ± 0.0011  0.140 ± 0.0043  0.137 ± 0.0029  0.143 ± 0.0051  0.134 ± 0.0030  0.136 ± 0.0031  0.134 ± 0.0007  0.149 ± 0.0018  0.192 ± 0.0021  0.195 ± 0.0009  0.189 ± 0.0039  0.193 ± 0.0024  0.189 ± 0.0037   
WFM-1 db4            0.136 ± 0.0032  0.141 ± 0.0021  0.134 ± 0.0016  0.134 ± 0.0034  0.136 ± 0.0005  0.130 ± 0.0018  0.127 ± 0.0019  0.129 ± 0.0020  0.123 ± 0.0018  0.189 ± 0.0017  0.192 ± 0.0005  0.185 ± 0.0017  0.188 ± 0.0022  0.184 ± 0.0011   
WFM-1 db6            0.138 ± 0.0009  0.143 ± 0.0045  0.139 ± 0.0049  0.136 ± 0.0017  0.136 ± 0.0002  0.124 ± 0.0018  0.128 ± 0.0011  0.134 ± 0.0004  0.120 ± 0.0055  0.190 ± 0.0011  0.196 ± 0.0020  0.191 ± 0.0031  0.191 ± 0.0014  0.183 ± 0.0011   
WFM-3 haar           0.143 ± 0.0015  0.144 ± 0.0052  0.139 ± 0.0026  0.136 ± 0.0040  0.133 ± 0.0049  0.127 ± 0.0014  0.123 ± 0.0029  0.132 ± 0.0005  0.118 ± 0.0025  0.194 ± 0.0013  0.199 ± 0.0020  0.188 ± 0.0013  0.190 ± 0.0005  0.181 ± 0.0029   
WFM-3 db2            0.150 ± 0.0023  0.146 ± 0.0019  0.145 ± 0.0032  0.148 ± 0.0010  0.141 ± 0.0066  0.136 ± 0.0039  0.123 ± 0.0031  0.141 ± 0.0025  0.123 ± 0.0025  0.193 ± 0.0013  0.196 ± 0.0008  0.191 ± 0.0015  0.201 ± 0.0019  0.195 ± 0.0052   
WFM-3 db4            0.151 ± 0.0028  0.146 ± 0.0028  0.141 ± 0.0041  0.144 ± 0.0037  0.146 ± 0.0038  0.137 ± 0.0037  0.132 ± 0.0053  0.140 ± 0.0023  0.131 ± 0.0052  0.195 ± 0.0014  0.197 ± 0.0022  0.189 ± 0.0029  0.192 ± 0.0018  0.189 ± 0.0009   
WFM-3 db6            0.145 ± 0.0020  0.147 ± 0.0046  0.144 ± 0.0034  0.139 ± 0.0055  0.141 ± 0.0021  0.125 ± 0.0022  0.132 ± 0.0035  0.136 ± 0.0054  0.119 ± 0.0013  0.193 ± 0.0008  0.198 ± 0.0019  0.188 ± 0.0016  0.191 ± 0.0029  0.181 ± 0.0025   
FM$_\text{pixel}$    0.144 ± 0.0035  0.142 ± 0.0032  0.142 ± 0.0025  0.143 ± 0.0025  0.141 ± 0.0029  0.143 ± 0.0027  0.137 ± 0.0039  0.154 ± 0.0038  0.126 ± 0.0053  0.193 ± 0.0021  0.199 ± 0.0016  0.191 ± 0.0003  0.196 ± 0.0016  0.186 ± 0.0025   
FM$_\text{fourier}$  0.242 ± 0.0029  0.245 ± 0.0018  0.228 ± 0.0002  0.249 ± 0.0026  0.238 ± 0.0025  0.229 ± 0.0010  0.229 ± 0.0035  0.243 ± 0.0018  0.217 ± 0.0018  0.496 ± 0.0027  0.497 ± 0.0011  0.493 ± 0.0013  0.502 ± 0.0022  0.493 ± 0.0014   
WDNO                 0.204 ± 0.0009  0.203 ± 0.0004  0.194 ± 0.0003  0.205 ± 0.0010  0.195 ± 0.0003  0.192 ± 0.0012  0.195 ± 0.0008  0.215 ± 0.0013  0.190 ± 0.0016  0.253 ± 0.0003  0.252 ± 0.0001  0.250 ± 0.0005  0.246 ± 0.0010  0.241 ± 0.0003   

                                                                                               High                                                                                                                                  
                               0.56            1.00            1.78            3.16            0.03            0.06            0.10            0.18            0.32            0.56            1.00            1.78            3.16  
WFM-1 haar           0.172 ± 0.0020  0.157 ± 0.0037  0.155 ± 0.0010  0.150 ± 0.0039  0.138 ± 0.0012  0.138 ± 0.0008  0.136 ± 0.0017  0.140 ± 0.0006  0.147 ± 0.0010  0.147 ± 0.0005  0.133 ± 0.0004  0.120 ± 0.0006  0.106 ± 0.0004  
WFM-1 db2            0.179 ± 0.0014  0.170 ± 0.0023  0.163 ± 0.0010  0.164 ± 0.0016  0.142 ± 0.0021  0.143 ± 0.0003  0.139 ± 0.002

In [11]:
display(view(P.s19a_spectral_ra(df, show_std=False)))  # Table S19 (Ra)
view(P.s19b_spectral_pr(df, show_std=False))          # Table S19 (Pr)

  wrote S19a_spectral_Ra


Low                                             Mid                                            High                                
                           13.815511       18.420681       23.025850       13.815511       18.420681       23.025850       13.815511       18.420681       23.025850
WFM-1 haar            0.174 ± 0.0017  0.166 ± 0.0078  0.138 ± 0.0030  0.189 ± 0.0015  0.135 ± 0.0014  0.133 ± 0.0002  0.110 ± 0.0016  0.186 ± 0.0055  0.163 ± 0.0013
WFM-1 db2             0.154 ± 0.0076  0.176 ± 0.0077  0.140 ± 0.0013  0.161 ± 0.0017  0.132 ± 0.0013  0.134 ± 0.0004  0.092 ± 0.0011  0.163 ± 0.0037  0.150 ± 0.0013
WFM-1 db4             0.160 ± 0.0211  0.169 ± 0.0061  0.140 ± 0.0027  0.121 ± 0.0059  0.130 ± 0.0005  0.131 ± 0.0002  0.049 ± 0.0020  0.102 ± 0.0008  0.120 ± 0.0006
WFM-1 db6             0.199 ± 0.0024  0.219 ± 0.0085  0.210 ± 0.0091  0.160 ± 0.0006  0.138 ± 0.0043  0.178 ± 0.0042  0.066 ± 0.0031  0.121 ± 0.0064  0.183 ± 0.0054
WFM-3 haar            0.154 ± 0.0050  0.163 ± 0.0047  0.130 ± 0.0016  0.188 ± 0.0038  0.133 ± 0.0008  0.129 ± 0.0005  0.110 ± 0.0030  0.181 ± 0.0021  0.156 ± 0.0022
WFM-3 db2             0.138 ± 0.0029  0.159 ± 0.0130  0.127 ± 0.0008  0.130 ± 0.0042  0.120 ± 0.0008  0.125 ± 0.0002  0.060 ± 0.0024  0.104 ± 0.0015  0.120 ± 0.0004
WFM-3 db4             0.137 ± 0.0101  0.160 ± 0.0063  0.130 ± 0.0037  0.118 ± 0.0048  0.119 ± 0.0007  0.125 ± 0.0003  0.052 ± 0.0018  0.097 ± 0.0027  0.120 ± 0.0008
WFM-3 db6             0.145 ± 0.0067  0.166 ± 0.0024  0.138 ± 0.0018  0.122 ± 0.0024  0.120 ± 0.0003  0.130 ± 0.0006  0.066 ± 0.0021  0.099 ± 0.0014  0.127 ± 0.0006
FM$_\text{pixel}$     0.155 ± 0.0022  0.150 ± 0.0019  0.131 ± 0.0005  0.188 ± 0.0015  0.130 ± 0.0018  0.128 ± 0.0003  0.112 ± 0.0006  0.183 ± 0.0060  0.156 ± 0.0027
FM$_\text{fourier}$   0.347 ± 0.0027  0.321 ± 0.0045  0.299 ± 0.0016  0.253 ± 0.0009  0.308 ± 0.0010  0.314 ± 0.0003  0.137 ± 0.0006  0.262 ± 0.0002  0.296 ± 0.0000
WDNO                  0.426 ± 0.0026  0.402 ± 0.0014  0.363 ± 0.0012  0.549 ± 0.0004  0.484 ± 0.0001  0.448 ± 0.0007  0.585 ± 0.0001  0.581 ± 0.0003  0.529 ± 0.0002
LatentFM$_{\div 64}$  0.206 ± 0.0049  0.183 ± 0.0033  0.167 ± 0.0022  0.177 ± 0.0012  0.149 ± 0.0004  0.151 ± 0.0003  0.111 ± 0.0012  0.139 ± 0.0010  0.196 ± 0.0001

  wrote S19b_spectral_Pr


Low                                             Mid                                            High                                
                           -2.302585        0.000000        2.302585       -2.302585        0.000000        2.302585       -2.302585        0.000000        2.302585
WFM-1 haar            0.187 ± 0.0073  0.144 ± 0.0035  0.147 ± 0.0018  0.158 ± 0.0020  0.164 ± 0.0002  0.136 ± 0.0001  0.192 ± 0.0035  0.155 ± 0.0009  0.111 ± 0.0018
WFM-1 db2             0.180 ± 0.0113  0.145 ± 0.0079  0.145 ± 0.0015  0.150 ± 0.0028  0.153 ± 0.0010  0.123 ± 0.0016  0.182 ± 0.0012  0.136 ± 0.0043  0.087 ± 0.0013
WFM-1 db4             0.181 ± 0.0261  0.141 ± 0.0058  0.148 ± 0.0028  0.138 ± 0.0040  0.133 ± 0.0022  0.111 ± 0.0004  0.120 ± 0.0015  0.087 ± 0.0018  0.063 ± 0.0005
WFM-1 db6             0.200 ± 0.0128  0.210 ± 0.0096  0.217 ± 0.0066  0.148 ± 0.0049  0.162 ± 0.0035  0.165 ± 0.0022  0.129 ± 0.0064  0.150 ± 0.0061  0.091 ± 0.0036
WFM-3 haar            0.171 ± 0.0014  0.139 ± 0.0033  0.137 ± 0.0010  0.155 ± 0.0029  0.159 ± 0.0017  0.136 ± 0.0016  0.189 ± 0.0038  0.150 ± 0.0038  0.108 ± 0.0018
WFM-3 db2             0.160 ± 0.0137  0.133 ± 0.0062  0.131 ± 0.0015  0.128 ± 0.0024  0.134 ± 0.0006  0.112 ± 0.0019  0.122 ± 0.0020  0.096 ± 0.0015  0.066 ± 0.0006
WFM-3 db4             0.163 ± 0.0064  0.130 ± 0.0043  0.134 ± 0.0004  0.126 ± 0.0010  0.128 ± 0.0025  0.108 ± 0.0013  0.112 ± 0.0022  0.090 ± 0.0013  0.066 ± 0.0005
WFM-3 db6             0.172 ± 0.0012  0.137 ± 0.0029  0.139 ± 0.0004  0.129 ± 0.0012  0.132 ± 0.0012  0.111 ± 0.0004  0.120 ± 0.0025  0.094 ± 0.0007  0.079 ± 0.0005
FM$_\text{pixel}$     0.169 ± 0.0036  0.132 ± 0.0020  0.136 ± 0.0021  0.153 ± 0.0008  0.160 ± 0.0016  0.133 ± 0.0010  0.197 ± 0.0014  0.149 ± 0.0069  0.106 ± 0.0012
FM$_\text{fourier}$   0.342 ± 0.0017  0.314 ± 0.0015  0.311 ± 0.0013  0.309 ± 0.0002  0.292 ± 0.0003  0.275 ± 0.0006  0.262 ± 0.0006  0.228 ± 0.0001  0.205 ± 0.0001
WDNO                  0.390 ± 0.0025  0.388 ± 0.0031  0.413 ± 0.0009  0.477 ± 0.0003  0.498 ± 0.0001  0.505 ± 0.0009  0.574 ± 0.0003  0.575 ± 0.0002  0.545 ± 0.0002
LatentFM$_{\div 64}$  0.199 ± 0.0106  0.173 ± 0.0045  0.185 ± 0.0017  0.160 ± 0.0012  0.160 ± 0.0002  0.157 ± 0.0002  0.163 ± 0.0014  0.151 ± 0.0006  0.132 ± 0.0002

## 6 · S20 kinetic energy, S21/S22 IC perturbation, S2 cross-scale coupling (+ RB/LatentFM summary, not in the paper)


In [12]:
view(P.x_rb_latentfm(df, show_std=False))            # rebuttal: RB + LatentFM

  wrote X_rb_latentfm


Spectral coherence RMSE                                            CRPS                                
                                         Low             Mid            High           01:32           33:64          65:196
WFM 1-scale                   0.168 ± 0.0049  0.118 ± 0.0009  0.077 ± 0.0002  0.007 ± 0.0000  0.041 ± 0.0003  0.073 ± 0.0032
WFM 3-scale                   0.143 ± 0.0020  0.109 ± 0.0009  0.075 ± 0.0006  0.004 ± 0.0001  0.032 ± 0.0003  0.069 ± 0.0011
FM$_\text{pixel}$             0.149 ± 0.0007  0.129 ± 0.0004  0.115 ± 0.0013  0.005 ± 0.0000  0.036 ± 0.0008  0.076 ± 0.0047
FM$_\text{fourier}$           0.327 ± 0.0011  0.263 ± 0.0001  0.209 ± 0.0002  0.009 ± 0.0001  0.038 ± 0.0001  0.096 ± 0.0004
WDNO                          0.392 ± 0.0007  0.474 ± 0.0002  0.535 ± 0.0001  0.050 ± 0.0000  0.102 ± 0.0002  0.138 ± 0.0003
LatentFM$_{\div 64}$          0.196 ± 0.0002  0.159 ± 0.0003  0.131 ± 0.0005  0.026 ± 0.0001  0.057 ± 0.0003  0.088 ± 0.0042

In [13]:
view(P.s20_ke_mae(gdf, show_std=False))                # rebuttal: kinetic energy

  wrote S20_ke_mae


turbulent_radiative_layer_2D                                 rayleigh_benard                                   active_matter                                
                                            01:32           33:64           65:97           01:32           33:64          65:196           01:16           17:32           33:77
WFM 1-scale                        0.047 ± 0.0014  0.113 ± 0.0018  0.221 ± 0.0059  0.007 ± 0.0001  0.410 ± 0.0048  0.379 ± 0.0103  0.502 ± 0.0010  0.538 ± 0.0047  0.508 ± 0.0015
WFM 3-scale                        0.017 ± 0.0008  0.067 ± 0.0027  0.147 ± 0.0040  0.003 ± 0.0001  0.217 ± 0.0056  0.348 ± 0.0227  0.112 ± 0.0018  0.391 ± 0.0107  0.531 ± 0.0034
FM$_\text{pixel}$                  0.016 ± 0.0010  0.067 ± 0.0041  0.177 ± 0.0066  0.004 ± 0.0000  0.235 ± 0.0050  0.251 ± 0.0101  0.051 ± 0.0010  0.288 ± 0.0117  0.320 ± 0.0098
FM$_\text{fourier}$                0.037 ± 0.0026  0.082 ± 0.0035  0.150 ± 0.0129  0.006 ± 0.0001  0.153 ± 0.0015  1.018 ± 0.0055  0.440 ± 0.0024  0.513 ± 0.0011  0.642 ± 0.0076
WDNO                               0.356 ± 0.0004  0.225 ± 0.0010  0.457 ± 0.0015  1.014 ± 0.0021  2.506 ± 0.0070  2.309 ± 0.0031  0.747 ± 0.0016  0.890 ± 0.0016  1.052 ± 0.0019
LatentFM$_{\div 64}$                           --              --              --  0.252 ± 0.0033  0.574 ± 0.0082  0.913 ± 0.0363              --              --              --

In [14]:
ic = P.s21_s22_ic_perturbation(show_std=False)             # rebuttal: IC perturbation
display(view(ic["crps"]))
view(ic["spectral"])

  wrote S21_ic_crps
  wrote S22_ic_spectral


,01:32,33:64,65:97
$\eta=0$,0.410 ± 0.0033,1.018 ± 0.0108,1.178 ± 0.0297
$\eta=0.001$,0.410 ± 0.0032,1.036 ± 0.0163,1.180 ± 0.0239
$\eta=0.01$,0.410 ± 0.0032,1.030 ± 0.0160,1.190 ± 0.0065
$\eta=0.1$,0.413 ± 0.0026,1.026 ± 0.0401,1.203 ± 0.0120


,Low,Mid,High
$\eta=0$,0.133 ± 0.0009,0.174 ± 0.0002,0.141 ± 0.0003
$\eta=0.001$,0.133 ± 0.0011,0.174 ± 0.0006,0.141 ± 0.0005
$\eta=0.01$,0.133 ± 0.0011,0.174 ± 0.0012,0.141 ± 0.0012
$\eta=0.1$,0.133 ± 0.0004,0.174 ± 0.0005,0.141 ± 0.0009


In [15]:
r = P.s2_cross_scale()   # from artifacts/tables/raw (written by 05_cross_scale_coupling.ipynb)
print(f"off-diagonal mass {r['off_diagonal_mass_pct']} %")
print((T.TABLES_TEX / "S2_cross_scale_sensitivity.tex").read_text())

  wrote cross-scale sensitivity (rho = 15.1 %)
off-diagonal mass 15.1 %
\begin{table}[ht]
  \centering
  \caption{Cross-scale sensitivity $S_{jj'}$ (Equation~\ref{eq:sensitivity}) of WFM 3-scale (haar) on TRL, with $\delta = 10^{-2}$, averaged over test windows and $\tau$. Rows: affected head $\hat{u}_j$; columns: perturbed scale $w^{\tau}_{j'}$ ($j=1$ finest). Independent per-scale flows would give zero off-diagonal entries; the measured off-diagonal mass is $\rho = 15.1\%$.}
  \label{tab:sensitivity}
  \begin{tabular}{lccc}
    \toprule
    & $w^{\tau}_1$ (fine) & $w^{\tau}_2$ (mid) & $w^{\tau}_3$ (coarse) \\
    \midrule
    $\hat{u}_1$ (fine) & 2.292 & 0.225 & 0.277 \\
    $\hat{u}_2$ (mid) & 0.135 & 2.366 & 0.311 \\
    $\hat{u}_3$ (coarse) & 0.095 & 0.183 & 2.243 \\
    \bottomrule
  \end{tabular}
\end{table}



## 7 · Checks

* **Seed spread** — relative std (%) of every cell, summarised per table.
* **Paper reproduction** — rebuilds the *submitted* paper's tables from the
  single-run files with the same builders; every cell should match.

In [16]:
#   P.write_seed_spread()

In [17]:
#   rep = paper_reproduction()
#   display(summary(rep))
#   assert rep["match"].all(), rep[~rep["match"]]
#   print(f"all {len(rep)} cells of the submitted paper reproduced")

## 8 · Files written

In [18]:
#   for d in (T.TABLES_TEX, T.TABLES_CSV, PROJECT_ROOT / "artifacts" / "checks"):
#    print(d.relative_to(PROJECT_ROOT), sorted(p.name for p in d.iterdir()))